## 13.05 全局向量的词嵌入（GloVe）


### 环境配置


In [1]:
import logging
logging.getLogger("matplotlib").setLevel(logging.WARNING)
logging.getLogger("torch_npu").setLevel(logging.WARNING)
import os
import sys
sys.path.insert(0, "..")
os.environ["TILE_FWK_DEVICE_ID"] = "0"
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    import pypto
    import torch
    from torch import nn
    import torch_npu

warnings.filterwarnings("ignore", message="Permission mismatch")
warnings.filterwarnings("ignore", message="TASK_QUEUE_ENABLE")
warnings.filterwarnings("ignore", message="On the interactive interface")
warnings.filterwarnings("ignore", message="Cannot create tensor")
import matplotlib.pyplot as plt

device_id = int(os.environ["TILE_FWK_DEVICE_ID"])
torch.npu.set_device(device_id)
device = f"npu:{device_id}"
pypto.pypto_impl.DeviceInit()

import math


### 练习 13.5.1

**题目：** 如果词 $w_i$ 和 $w_j$ 在同一上下文窗口中同时出现，我们如何使用它们在文本序列中的距离来重新设计计算条件概率 $p_{ij}$ 的方法？提示：参见 GloVe 论文的第 4.2 节。

**解答：** GloVe 论文第 4.2 节给出了基于距离衰减的加权方案。设两个词在文本序列中的距离为 $d$，则共现计数可加权为

$$X_{ij} = \sum_{k} \frac{1}{d_{ij,k}},$$

即距离越近的共现贡献越大（$1/d$ 衰减），条件概率 $p_{ij} = X_{ij} / X_i$ 也随之改变。论文给出的通用加权函数为

$$X_{ij} = \sum_{k} f(d_{ij,k}), \quad f(d) = \begin{cases} (1 - d/\text{window})^{\alpha}, & d \le \text{window} \\ 0, & \text{否则} \end{cases}$$

（当 $\alpha = 1$、窗口无限大时退化为普通的共现计数）。这使模型能区分“紧邻共现”与“相隔较远共现”对语义关系的不同贡献。


### 练习 13.5.2

**题目：** 对于任何一个词，它的中心词偏置和上下文偏置在数学上是等价的吗？为什么？

**解答：** 在数学形式上两者**等价**。GloVe 的损失为

$$\sum_{i,j} f(X_{ij}) \left(\mathbf{w}_i^\top \tilde{\mathbf{w}}_j + b_i + \tilde{b}_j - \log X_{ij}\right)^2,$$

目标关于 $(\mathbf{w}_i, b_i)$ 与 $(\tilde{\mathbf{w}}_i, \tilde{b}_i)$ 完全对称：把中心词嵌入与上下文词嵌入互换、两个偏置互换，损失函数形式不变。因此一个词的“中心词偏置”与“上下文偏置”只是同一优化问题中对称的两个角色，在数学上可以互换，最终学到的向量也往往互为近似转置。

（注意：**数值上**两者通常不严格相等——训练使用随机初始化且两组参数独立更新，但它们的语义角色是对称的；若模型训练到全局最优且目标函数关于两者对称，则存在互换后的等价解。）


---

## 参考答案来源
参考答案和 PyTorch 代码实现来源：[https://datawhalechina.github.io/d2l-ai-solutions-manual/#/](https://datawhalechina.github.io/d2l-ai-solutions-manual/#/)
